# Exercício 2 — Duplicatas
**Disciplina:** Ciência de Dados · **Etapa CRISP-DM:** Preparação dos Dados
**Dataset:** KPMG Customer Transactions (`KPMG_VI_New_raw_data_update_final.xlsx`, planilha *Transactions*)

**Objetivo de negócio:** garantir que cada transação seja contada uma única vez antes de calcular faturamento, ticket médio e produtos mais vendidos.

In [1]:
# IMPORTAÇÕES
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)

In [2]:
# CARREGAMENTO DOS DADOS
arquivo = 'KPMG_VI_New_raw_data_update_final.xlsx'

# Primeira tentativa: leitura "padrão"
bruto = pd.read_excel(arquivo, sheet_name='Transactions')
bruto.head(4)

,Note: The data and information in this document is reflective of a hypothetical situation and client. This document is to be used for KPMG Virtual Internship purposes only.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
1,1,2,2950,2017-02-25 00:00:00,False,Approved,Solex,Standard,medium,medium,71.49,53.62,41245
2,2,3,3120,2017-05-21 00:00:00,True,Approved,Trek Bicycles,Standard,medium,large,"2,091.47",388.92,41701
3,3,37,402,2017-10-16 00:00:00,False,Approved,OHM Cycles,Standard,low,medium,"1,793.43",248.82,36361


## Conferência do cabeçalho e dos tipos
**Problema encontrado:** o cabeçalho **não foi lido corretamente**. A primeira linha da planilha é uma nota de rodapé do KPMG ("Note: The data and information in this document is reflective of a hypothetical situation..."), então o pandas usou esse texto como nome da primeira coluna e todas as outras viraram `Unnamed: n`. Os nomes reais (`transaction_id`, `product_id`...) estão na **segunda linha** (índice 1). Basta ajustar com `header=1`.

In [3]:
# CONFERÊNCIA DO CABEÇALHO E DOS TIPOS
df = pd.read_excel(arquivo, sheet_name='Transactions', header=1)

print('Dimensões:', df.shape)
print()
print(pd.DataFrame({'tipo': df.dtypes.astype(str), 'ausentes': df.isna().sum()}))
df.head()

Dimensões: (20000, 13)

                                   tipo  ausentes
transaction_id                    int64         0
product_id                        int64         0
customer_id                       int64         0
transaction_date         datetime64[us]         0
online_order                    float64       360
order_status                        str         0
brand                               str       197
product_line                        str       197
product_class                       str       197
product_size                        str       197
list_price                      float64         0
standard_cost                   float64       197
product_first_sold_date         float64       197


,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
0,1,2,2950,2017-02-25,0.00,Approved,Solex,Standard,medium,medium,71.49,53.62,"41,245.00"
1,2,3,3120,2017-05-21,1.00,Approved,Trek Bicycles,Standard,medium,large,"2,091.47",388.92,"41,701.00"
2,3,37,402,2017-10-16,0.00,Approved,OHM Cycles,Standard,low,medium,"1,793.43",248.82,"36,361.00"
3,4,88,3135,2017-08-31,0.00,Approved,Norco Bicycles,Standard,medium,medium,"1,198.46",381.10,"36,145.00"
4,5,78,787,2017-10-01,1.00,Approved,Giant Bicycles,Standard,medium,large,"1,765.30",709.48,"42,226.00"


**Observações sobre os tipos** (não alterei nada ainda, só registro):
- `online_order` veio como `float` (0.0/1.0 com ausentes) porque é um booleano com valores faltando.
- `product_first_sold_date` veio como número (41245, 41701...), que é uma **data em formato serial do Excel**, não uma quantidade.
- `brand`, `product_line`, `product_class`, `product_size`, `standard_cost` e `product_first_sold_date` têm os mesmos **197 ausentes**: são as mesmas linhas, provavelmente transações sem produto cadastrado.

## Duplicatas exatas

In [4]:
# DUPLICATAS EXATAS
n_exatas = df.duplicated().sum()
print('Linhas completamente iguais:', n_exatas)

Linhas completamente iguais: 0


**Resultado:** nenhuma duplicata exata. Mas isso **não encerra o assunto** (ver reflexão 1): como cada linha tem seu próprio `transaction_id`, duas linhas que descrevem a mesma compra jamais seriam completamente iguais.

## Duplicatas por subconjunto de colunas
**Critério escolhido para "mesma transação":** `customer_id` + `product_id` + `transaction_date`. Justificativa: uma compra real é caracterizada por **quem** comprou, **o quê** e **quando**. Não incluo `transaction_id` (ele é o campo que pode ter sido gerado de novo em um reenvio) nem `list_price` (um preço divergente pode ser justamente a diferença entre uma repetição e uma venda real; testo as duas versões abaixo).

In [5]:
# DUPLICATAS POR SUBCONJUNTO DE COLUNAS
subconjuntos = {
    'cliente + produto + data': ['customer_id', 'product_id', 'transaction_date'],
    '... + list_price': ['customer_id', 'product_id', 'transaction_date', 'list_price'],
    'tudo menos transaction_id': [c for c in df.columns if c != 'transaction_id'],
}
res = pd.DataFrame({
    nome: {'linhas em grupos repetidos': df.duplicated(cols, keep=False).sum(),
           'registros "a mais"': df.duplicated(cols, keep='first').sum()}
    for nome, cols in subconjuntos.items()
}).T
res

,linhas em grupos repetidos,"registros ""a mais"""
cliente + produto + data,2,1
... + list_price,0,0
tudo menos transaction_id,0,0


In [6]:
# Examinando o grupo repetido encontrado pelo critério cliente + produto + data
crit = ['customer_id', 'product_id', 'transaction_date']
df[df.duplicated(crit, keep=False)].sort_values(crit)

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
10910,10911,0,1840,2017-07-11,0.00,Approved,OHM Cycles,Standard,medium,medium,235.63,125.07,"36,367.00"
16026,16027,0,1840,2017-07-11,0.00,Approved,Solex,Standard,medium,large,202.62,151.96,"33,364.00"


Há **um único par** com o mesmo cliente, produto e data (cliente 1840, `product_id` 0, 11/07/2017). Mas as duas linhas têm `transaction_id` diferentes, **marcas diferentes** (OHM Cycles x Solex), **tamanhos diferentes** e `list_price` diferente. São, na prática, **dois produtos distintos** que compartilham o mesmo `product_id = 0`, e não uma repetição. O problema aqui é o identificador do produto (ver próxima seção), não uma duplicata. Mantenho como **caso sinalizado para o negócio**, não como duplicata a remover (ver estratégia abaixo).

## Unicidade dos identificadores

In [7]:
# UNICIDADE DOS IDENTIFICADORES
ids = pd.DataFrame({
    'valores_unicos': df[['transaction_id', 'product_id', 'customer_id']].nunique(),
    'linhas': len(df),
})
ids['repetidos'] = ids['linhas'] - ids['valores_unicos']
print(ids)
print()
print('transaction_id é único?', df['transaction_id'].is_unique)
print('transaction_id vai de', df['transaction_id'].min(), 'a', df['transaction_id'].max(), '| sequência completa?',
      set(df['transaction_id']) == set(range(df['transaction_id'].min(), df['transaction_id'].max() + 1)))

# product_id deveria "amarrar" um produto: ele tem sempre a mesma marca e o mesmo preço?
por_produto = df.groupby('product_id').agg(marcas=('brand', 'nunique'), precos=('list_price', 'nunique'), linhas=('transaction_id', 'size'))
print('\nproduct_id com mais de 1 marca:', (por_produto['marcas'] > 1).sum(), 'de', len(por_produto))
print('product_id com mais de 1 list_price:', (por_produto['precos'] > 1).sum(), 'de', len(por_produto))
por_produto.sort_values('precos', ascending=False).head(5)

                valores_unicos  linhas  repetidos
transaction_id           20000   20000          0
product_id                 101   20000      19899
customer_id               3494   20000      16506

transaction_id é único? True
transaction_id vai de 1 a 20000 | sequência completa? True

product_id com mais de 1 marca: 63 de 101
product_id com mais de 1 list_price: 77 de 101


,marcas,precos,linhas
product_id,,,
0,6,221,1378
1,1,2,311
2,2,2,240
4,2,2,241
5,2,2,222


**Conclusões sobre identificadores:**
- `transaction_id` **é realmente único** (20.000 valores em 20.000 linhas, sequência 1 a 20.000 sem buracos).
- `product_id`, `customer_id` se repetem (esperado: um cliente compra várias vezes e um produto é vendido várias vezes).
- Um alerta importante: **63 dos 101 `product_id` aparecem com mais de uma marca** e 77 com mais de um `list_price`. O caso extremo é o `product_id = 0`: 1.378 transações, **6 marcas** e **221 preços diferentes**. Isso indica que o 0 funciona como **código padrão/placeholder**, e não como um produto real. Então `product_id` sozinho não identifica um produto de forma confiável. Não é uma duplicata, mas é uma inconsistência que deve ser reportada ao dono dos dados antes de qualquer análise "por produto" (por exemplo, "produtos mais vendidos").

## Simulação: por que "zero duplicatas exatas" não basta
Como o dataset real quase não tem repetições, faço uma **simulação sobre uma cópia** (a base original não é alterada): reenvio 100 transações como se um sistema tivesse duplicado o lançamento, mudando só o `transaction_id`. É assim que duplicatas aparecem na vida real.

In [8]:
# SIMULAÇÃO (apenas demonstração — trabalha em cópia)
rng = np.random.default_rng(7)
reenvio = df.sample(100, random_state=7).copy()
reenvio['transaction_id'] = np.arange(df['transaction_id'].max() + 1, df['transaction_id'].max() + 101)
simulada = pd.concat([df, reenvio], ignore_index=True)

print('Duplicatas EXATAS na base simulada:        ', simulada.duplicated().sum())
print('Duplicatas por subconjunto (sem o id):     ', simulada.duplicated([c for c in simulada.columns if c != 'transaction_id']).sum())
print(f"Soma de list_price: real {df['list_price'].sum():,.2f} -> simulada {simulada['list_price'].sum():,.2f}  (inflada em {simulada['list_price'].sum() - df['list_price'].sum():,.2f})")

Duplicatas EXATAS na base simulada:         0


Duplicatas por subconjunto (sem o id):      100
Soma de list_price: real 22,156,588.98 -> simulada 22,273,709.20  (inflada em 117,120.22)


## Estratégia de tratamento e impacto
**Decisões**
1. **Duplicatas exatas:** nenhuma, nada a remover.
2. **Par cliente + produto + data (o único achado):** `transaction_id`, marca, tamanho e `list_price` são diferentes; não há evidência de repetição. **Mantenho os dois registros** e deixo o caso sinalizado numa coluna `suspeita_duplicata`, para que o negócio decida. Calculo também o impacto da alternativa (remover o mais antigo e manter o mais recente, `keep='last'`) para mostrar o quanto isso afetaria os indicadores.
3. **Regra preparada para o futuro** (já demonstrada na simulação): sempre rodar `duplicated()` sobre o subconjunto **sem** o identificador gerado pelo sistema.

In [9]:
# ESTRATÉGIA DE TRATAMENTO E IMPACTO
df_limpo = df.copy()
df_limpo['suspeita_duplicata'] = df_limpo.duplicated(crit, keep=False)

# Alternativa: manter apenas o registro mais recente (maior transaction_id) de cada grupo
df_alt = df.sort_values('transaction_id').drop_duplicates(crit, keep='last')

def indicadores(base):
    return pd.Series({
        'registros': len(base),
        'soma list_price': base['list_price'].sum(),
        'ticket médio (list_price)': base['list_price'].mean(),
        'soma lucro (list_price - standard_cost)': (base['list_price'] - base['standard_cost']).sum(),
    })

impacto = pd.DataFrame({
    'original / mantendo os dois': indicadores(df_limpo),
    'removendo repetição (keep=last)': indicadores(df_alt),
    'simulação com 100 duplicatas': indicadores(simulada),
})
impacto['diferença (keep=last)'] = impacto['removendo repetição (keep=last)'] - impacto['original / mantendo os dois']
impacto['diferença (simulação)'] = impacto['simulação com 100 duplicatas'] - impacto['original / mantendo os dois']
print('Registros marcados como suspeitos:', df_limpo['suspeita_duplicata'].sum(), '| removidos na alternativa:', len(df) - len(df_alt))
impacto

Registros marcados como suspeitos:

 2 | removidos na alternativa: 1


,original / mantendo os dois,removendo repetição (keep=last),simulação com 100 duplicatas,diferença (keep=last),diferença (simulação)
registros,"20,000.00","19,999.00","20,100.00",-1.00,100.00
soma list_price,"22,156,588.98","22,156,353.35","22,273,709.20",-235.63,"117,120.22"
ticket médio (list_price),"1,107.83","1,107.87","1,108.14",0.04,0.32
soma lucro (list_price - standard_cost),"10,930,283.96","10,930,173.40","10,985,537.10",-110.56,"55,253.14"


**Impacto:** na base real, tratar o par como duplicata removeria **1 registro** e reduziria a soma de `list_price` em **235,63** de ~22,16 milhões (≈ 0,001%), diferença desprezível. Logo, a decisão de mantê-lo é mais de **princípio** (não apagar dados sem evidência) do que de materialidade. Já na simulação, 100 reenvios (0,5% das linhas) inflam a soma de `list_price` e o lucro na mesma proporção e **passariam despercebidos** por um `duplicated()` simples, que acusou zero.

## Desafio 1 — Textos parecidos com `rapidfuzz`
Duplicatas nem sempre são idênticas. Uso o `rapidfuzz` para buscar valores parecidos em colunas de texto.

In [10]:
# DESAFIO — rapidfuzz em colunas de texto reais
def pares_parecidos(serie, limite=85):
    valores = serie.dropna().astype(str).unique()
    saida = []
    for i, a in enumerate(valores):
        for b in valores[i + 1:]:
            score = fuzz.token_sort_ratio(a.lower(), b.lower())
            if score >= limite:
                saida.append((a, b, round(score, 1)))
    return pd.DataFrame(saida, columns=['valor_a', 'valor_b', 'similaridade']).sort_values('similaridade', ascending=False)

print('Marcas distintas em Transactions:', df['brand'].dropna().unique().tolist())
print('Pares de marcas com similaridade >= 85:')
print(pares_parecidos(df['brand']).to_string(index=False) or 'nenhum')

Marcas distintas em Transactions: ['Solex', 'Trek Bicycles', 'OHM Cycles', 'Norco Bicycles', 'Giant Bicycles', 'WeareA2B']
Pares de marcas com similaridade >= 85:
Empty DataFrame
Columns: [valor_a, valor_b, similaridade]
Index: []


In [11]:
# Mesma busca em uma coluna com muito mais variação de texto: job_title (planilha CustomerDemographic)
demo = pd.read_excel(arquivo, sheet_name='CustomerDemographic')
print('job_title distintos:', demo['job_title'].nunique())
pj = pares_parecidos(demo['job_title'], limite=88)
print('Pares com similaridade >= 88:', len(pj))
pj.head(15)

job_title distintos: 195
Pares com similaridade >= 88: 169


,valor_a,valor_b,similaridade
22,Budget/Accounting Analyst I,Budget/Accounting Analyst II,98.20
23,Budget/Accounting Analyst I,Budget/Accounting Analyst IV,98.20
93,Budget/Accounting Analyst II,Budget/Accounting Analyst III,98.20
132,Computer Systems Analyst II,Computer Systems Analyst III,98.20
125,Administrative Assistant III,Administrative Assistant II,98.20
112,Human Resources Assistant II,Human Resources Assistant III,98.20
56,Human Resources Assistant I,Human Resources Assistant II,98.20
57,Human Resources Assistant I,Human Resources Assistant IV,98.20
16,Computer Systems Analyst I,Computer Systems Analyst II,98.10
99,Administrative Assistant IV,Administrative Assistant I,98.10


In [12]:
# Demonstração: a técnica pega "Giant Bicycles" x "Giant Bicycle" quando o erro existe.
# Crio uma cópia da coluna brand com erros de digitação artificiais (original intacto).
sujo = df['brand'].copy()
idx = sujo[sujo == 'Giant Bicycles'].sample(300, random_state=1).index
sujo.loc[idx[:150]] = 'Giant Bicycle'
sujo.loc[idx[150:]] = 'giant  bicycles '
print('Antes da correção — marcas distintas:', sujo.dropna().nunique())
print(pares_parecidos(sujo).to_string(index=False))

# Correção: mapear cada variante para a forma canônica mais frequente do seu grupo
canonicas = df['brand'].dropna().unique().tolist()
def canonizar(txt):
    if pd.isna(txt):
        return txt
    melhor, score, _ = process.extractOne(' '.join(txt.split()).lower(), [c.lower() for c in canonicas], scorer=fuzz.token_sort_ratio)
    return next(c for c in canonicas if c.lower() == melhor) if score >= 85 else txt

corrigido = sujo.map(canonizar)
print('\nDepois da correção — marcas distintas:', corrigido.dropna().nunique())
print((corrigido == df['brand']).mean().round(4), '<- fração de linhas idêntica ao original (NaN conta como diferente)')

Antes da correção — marcas distintas: 8
         valor_a          valor_b  similaridade
  Giant Bicycles giant  bicycles         100.00
  Giant Bicycles    Giant Bicycle         96.30
giant  bicycles     Giant Bicycle         96.30

Depois da correção — marcas distintas: 6
0.9902 <- fração de linhas idêntica ao original (NaN conta como diferente)


**Achados do desafio:**
- Em `brand` real **não há variantes parecidas** (nenhum par ≥ 85), então a coluna já está padronizada.
- Em `job_title`, a busca fuzzy com limiar 88 devolve **169 pares**, mas os primeiros são "Budget/Accounting Analyst I" x "II" x "IV", "Human Resources Assistant I" x "II"... Eles têm similaridade de ~98%, mas são **níveis hierárquicos diferentes**, e não o mesmo cargo escrito de forma diferente. Ou seja, são **falsos positivos**. Fuzzy matching **sugere candidatos, não decide**: é preciso conferir à mão, e o limiar (85, 88...) é uma decisão que precisa ser justificada. Uma alternativa seria comparar o texto sem numerais romanos para separar "variação de grafia" de "nível diferente".
- Na demonstração com erros artificiais, a técnica agrupa de volta `Giant Bicycle` e `giant  bicycles ` em `Giant Bicycles`.

## Desafio 2 — Gênero na planilha CustomerDemographic

In [13]:
# DESAFIO — formas diferentes de registrar gênero
print('Valores brutos (sem tratamento):')
print(demo['gender'].value_counts(dropna=False))

normal = demo['gender'].astype(str).str.strip().str.lower()
print('\nApós strip + lower:')
print(normal.value_counts(dropna=False))
print('\nFormas distintas de escrever gênero:', demo['gender'].nunique(dropna=False))

Valores brutos (sem tratamento):
gender
Female    2039
Male      1873
U           88
Name: count, dtype: int64

Após strip + lower:
gender
female    2039
male      1873
u           88
Name: count, dtype: int64

Formas distintas de escrever gênero: 3


**Resultado:** nesta versão do arquivo a coluna `gender` tem **3 formas**: `Female`, `Male` e `U`. Nenhuma variação de grafia (como "F", "Femal" ou "M"), então a padronização por `strip`/`lower` não altera nada. O que merece atenção é o **`U`**: significa *Unknown* (não informado) e deve ser tratado como **ausente**, não como uma terceira categoria de gênero (`demo['gender'].replace('U', np.nan)`), ou mantido como categoria própria, mas **de forma consciente**. Versões originais desse dataset circulam com variações como `F`, `Femal`, `M`, e a verificação acima é a que detectaria isso.

## Respostas às perguntas de reflexão

**1. Se o resultado de duplicatas exatas for zero, isso garante que não há registros repetidos? Por quê?**
Não. `duplicated()` só reconhece linhas **idênticas em todas as colunas**. Se o sistema gera um identificador novo a cada lançamento (como o `transaction_id`), a mesma compra reenviada fica com id diferente e **escapa** da checagem. Na simulação acima, a base com 100 reenvios tem **0 duplicatas exatas** e, ainda assim, 100 transações contadas duas vezes. Também há casos de diferenças sutis (espaço extra, maiúscula/minúscula, "Bicycle" x "Bicycles", preço arredondado). Por isso o procedimento correto é verificar também **por subconjunto de colunas** e por **similaridade de texto**.

**2. Em uma base de transações, duas linhas idênticas são sempre um erro? Dê um exemplo em que não seriam.**
Não. Exemplo: um cliente compra, no mesmo dia, **duas unidades do mesmo produto** em dois lançamentos separados do caixa (ou um mercado registra "1 item" duas vezes porque o cliente pegou dois), sem campo de quantidade nem horário na base. As linhas ficam idênticas e ambas são **vendas reais**. Remover uma subestimaria o faturamento. Outro caso: assinaturas mensais de mesmo valor lançadas em linhas iguais por falta de coluna de competência. Se a granularidade da base não distingue os eventos, a igualdade é um limite da modelagem, não um erro de digitação.

**3. Por que a escolha das colunas que definem "o mesmo registro" é uma decisão de negócio, e não apenas técnica? Relacione com o Entendimento do Negócio do CRISP-DM.**
Porque "o mesmo registro" é uma **definição do negócio**: o que conta como um evento único? Se eu uso `cliente + produto + data`, estou dizendo que um cliente nunca compra o mesmo produto duas vezes no mesmo dia. Isso só o negócio pode confirmar. Se incluo `list_price`, estou aceitando que o mesmo item com preço diferente é outra venda. Colunas demais deixam repetições reais escaparem; colunas de menos apagam vendas legítimas. Na etapa de **Entendimento do Negócio** (a primeira do CRISP-DM) se define o que se quer medir (faturamento? número de clientes? produtos mais vendidos?) e qual a granularidade correta dos dados. É desse entendimento que sai a regra de unicidade, e o código só a executa. Por isso registrei a decisão e o impacto nos indicadores, em vez de simplesmente rodar `drop_duplicates()`.